# Chapter 8 &mdash; Solving the Stamp Problem with a Minimal DFA: $Fr = |Q| - 2$

**Concept 11 of the Chapter 8 decomposition:** *Solving the Stamp Problem with a Minimal DFA: $Fr = |Q| - 2$*

Model a $p$-cent stamp as $1^p$; the minimal DFA for $(1^p+1^q)^*$ is a lasso whose stem length gives $Fr$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Stamp-Problem-By-DFA/Concept-Stamp-Problem-By-DFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Here is the chapter's most satisfying result. Model a $p$-cent stamp as the string
$1^p$. Then the payable amounts are exactly the lengths of strings in
$(1^p + 1^q)^*$.

Build the **minimal DFA** for that expression. Because the alphabet is a single
letter, the machine is a **lasso**: a stem of unpayable amounts, then a cycle where
everything is payable. And the arithmetic falls out:

$$Fr(p,q) = |Q| - 2.$$

For $p=3,q=5$: the minimal DFA has 9 states, and $9-2=7=Fr(3,5)$. An automaton has
computed a number-theoretic quantity.

## 2. Definitions

### The construction

In [ ]:
def stamp_dfa(p, q):
    return min_dfa(nfa2dfa(re2nfa("(%s+%s)*" % ('1'*p, '1'*q))))

def fr_from_dfa(p, q): return len(stamp_dfa(p, q)["Q"]) - 2

### The two references

In [ ]:
from math import gcd
def sylvester(p, q): return p*q - p - q if gcd(p, q) == 1 else None

def payable_set(p, q, upto):
    return {p*i + q*j for i in range(upto//p + 1) for j in range(upto//q + 1)
            if p*i + q*j <= upto}

## 3. Tests

For $p=3,q=5$ the minimal DFA has 9 states, and $9-2=7$.

In [ ]:
D = stamp_dfa(3, 5)
print("|Q| =", len(D["Q"]))
print("Fr from the DFA :", fr_from_dfa(3, 5))
print("Sylvester       :", sylvester(3, 5))
assert fr_from_dfa(3, 5) == sylvester(3, 5) == 7

The formula holds across coprime pairs.

In [ ]:
print("%-10s %-8s %-12s %s" % ("(p,q)", "|Q|", "|Q|-2", "pq-p-q"))
for p, q in [(3,5), (3,7), (4,7), (5,7), (5,9), (7,11)]:
    if gcd(p, q) != 1: continue
    D = stamp_dfa(p, q)
    print("%-10s %-8d %-12d %d" % ("(%d,%d)" % (p,q), len(D["Q"]),
                                   len(D["Q"]) - 2, sylvester(p, q)))
    assert len(D["Q"]) - 2 == sylvester(p, q)

The DFA really does accept exactly the payable amounts.

In [ ]:
p, q = 3, 5
D = stamp_dfa(p, q)
pay = payable_set(p, q, 40)
mismatch = [n for n in range(41) if accepts_dfa(D, '1'*n) != (n in pay)]
print("mismatches up to 40 :", mismatch)
assert not mismatch
print("unpayable :", [n for n in range(41) if not accepts_dfa(D, '1'*n)])

**The lasso:** a stem of unpayable amounts, then a cycle where everything works.

In [ ]:
seq = [run_dfa(D, '1'*n) for n in range(14)]
seen, stem = {}, None
for i, st in enumerate(seq):
    if st in seen: stem = (seen[st], i); break
    seen[st] = i
print("state sequence repeats: first visit %d, second visit %d" % stem)
print("stem length %d, cycle length %d" % (stem[0], stem[1] - stem[0]))
print("\nThe stem holds the unpayable amounts; Fr is the last of them.")

Non-coprime denominations: the machine is **not** a lasso with a full cycle.

In [ ]:
D46 = stamp_dfa(4, 6)
print("(4,6) minimal |Q| =", len(D46["Q"]), " |Q|-2 =", len(D46["Q"]) - 2)
print("but Sylvester says :", sylvester(4, 6))
print("accepted lengths :", [n for n in range(25) if accepts_dfa(D46, '1'*n)])
print("\nOdd amounts are never payable, so there is no largest gap and |Q|-2 is")
print("not a Frobenius number.  The formula needs gcd(p,q) = 1.")

## 4. Animation

The lasso for $(1^3+1^5)^*$ &mdash; count the stem states.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(stamp_dfa(3, 5), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Draw the lasso for $(1^2+1^5)^*$. What is the stem length?
2. Why is the $-2$ there? Which two states does it account for?
3. Try three denominations. Does $|Q|-2$ still give the Frobenius number?

In [ ]:
# Your work for the exercises above.